In [1]:
import yfinance as yf
import pandas as pd

print(f"yfinance version: {yf.__version__}")

yfinance version: 1.3.0


In [2]:
ticker_list = ['2267.T', '3435.T', '1846.HK', '6889.HK', '7399.T', '1913.HK', 'VTU.L']


def pull_yf_ticker_data(ticker_list):
    extracted_data = []

    for ticker_str in ticker_list:
        print(f"Fetching data for: {ticker_str}...")
        try:
            ticker = yf.Ticker(ticker_str)
            info = ticker.info

            # --- FIX 1: Guard .upper() calls against None ---
            # info.get() can return None; calling .upper() on None raises AttributeError
            # and silently drops the entire ticker into the except block.
            raw_trading_curr = info.get("currency", None)
            raw_financial_curr = info.get("financialCurrency", None)
            trading_curr = raw_trading_curr.upper() if raw_trading_curr else None
            financial_curr = raw_financial_curr.upper() if raw_financial_curr else None

            market_cap = info.get("marketCap", None)  # Keep as None if missing

            # --- FX conversion: only run when both currencies are known and differ ---
            market_cap_converted = market_cap
            if trading_curr and financial_curr and trading_curr != financial_curr:
                fx_ticker_str = f"{trading_curr}{financial_curr}=X"
                print(
                    f"   ⚠️  Currency mismatch ({trading_curr} vs {financial_curr}). "
                    f"Fetching FX rate {fx_ticker_str}..."
                )
                try:
                    fx_data = yf.Ticker(fx_ticker_str).history(period="1d")
                    if not fx_data.empty:
                        fx_rate = fx_data["Close"].iloc[-1]
                        if market_cap is not None:
                            market_cap_converted = market_cap * fx_rate
                            print(f"   Converted Market Cap at rate: {fx_rate:.4f}")
                    else:
                        print(f"   Could not fetch FX data for {fx_ticker_str}. Using original.")
                except Exception as fx_err:
                    print(f"   FX fetch error: {fx_err}")

            # --- Balance Sheet ---
            q_bs = ticker.quarterly_balance_sheet
            total_assets = current_assets = total_liabilities = None
            total_investments = total_goodwill_intangibles = None

            if not q_bs.empty:
                latest_col = q_bs[q_bs.columns[0]]
                total_assets = latest_col.get("Total Assets")
                current_assets = latest_col.get("Current Assets")
                total_liabilities = latest_col.get("Total Liabilities Net Minority Interest")
                total_goodwill_intangibles = latest_col.get("Goodwill And Other Intangible Assets")

                # --- FIX 2: Investment Properties is almost never reported ---
                # Use "Investments And Advances" as a broader fallback, which
                # covers long-term investments more reliably across exchange filings.
                total_investments = latest_col.get(
                    "Investment Properties",
                    latest_col.get("Investments And Advances")  # fallback
                )

            # --- Annual Net Income (latest FY) ---
            annual_inc = ticker.income_stmt
            latest_fy_net_inc = (
                annual_inc.loc["Net Income"].iloc[0]
                if not annual_inc.empty and "Net Income" in annual_inc.index
                else None
            )

            # --- FIX 3: TTM income — correct parameter is `mode` (singular) ---
            # `modes` (plural) was the old name; it still works in some versions
            # but generates a FutureWarning and may silently return nothing.
            ttm_net_inc = None
            try:
                ttm_inc = ticker.get_income_stmt(mode="trailing")  # singular `mode`
                if not ttm_inc.empty and "Net Income" in ttm_inc.index:
                    ttm_net_inc = ttm_inc.loc["Net Income"].iloc[0]
            except Exception:
                pass

            # Fallback: sum last 4 quarters if TTM endpoint returned nothing
            if ttm_net_inc is None:
                q_inc = ticker.quarterly_income_stmt
                if not q_inc.empty and "Net Income" in q_inc.index:
                    ttm_net_inc = q_inc.loc["Net Income"].iloc[:4].sum()

            extracted_data.append({
                "Ticker": ticker_str,
                "Trading Currency": trading_curr,
                "Financial Currency": financial_curr,
                "Market Cap": market_cap_converted,   # None if unavailable
                "Total Assets": total_assets,
                "Total Current Assets": current_assets,
                "Total Goodwill and Intangibles": total_goodwill_intangibles,
                "Total Liabilities": total_liabilities,
                "Total Investments": total_investments,
                "Latest FY Net Income": latest_fy_net_inc,
                "TTM Net Income": ttm_net_inc,
            })

        except Exception as e:
            print(f"Error fetching {ticker_str}: {e}")
            extracted_data.append({
                "Ticker": ticker_str,
                "Trading Currency": None, "Financial Currency": None,
                "Market Cap": None, "Total Assets": None,
                "Total Current Assets": None, "Total Goodwill and Intangibles": None,
                "Total Liabilities": None, "Total Investments": None,
                "Latest FY Net Income": None, "TTM Net Income": None,
            })

    return pd.DataFrame(extracted_data)

In [3]:
def format_currency(val):
    """Formats large numbers into human-readable strings. Returns 'N/A' for missing data."""
    # --- FIX 4: Guard against NaN/None before arithmetic ---
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return "N/A"
    abs_val = abs(val)
    if abs_val >= 1_000_000_000_000:
        return f"{val / 1_000_000_000_000:.1f}t"
    elif abs_val >= 1_000_000_000:
        return f"{val / 1_000_000_000:.1f}b"
    elif abs_val >= 1_000_000:
        return f"{val / 1_000_000:.1f}m"
    elif abs_val >= 1_000:
        return f"{val / 1_000:.1f}k"
    return str(int(val)) if val == int(val) else f"{val:.1f}"


def format_ratio(val):
    """Returns 'N/A' for negative, infinite, or NaN ratios, otherwise 2 decimals."""
    if pd.isna(val) or val < 0 or val in (float("inf"), float("-inf")):
        return "N/A"
    return f"{val:.2f}"

In [4]:
df = pull_yf_ticker_data(ticker_list=ticker_list)

formatted_df = (
    df
    .assign(
        # Fill investments with 0 only — absence = no investment properties held
        Total_Investments_filled=lambda d: d["Total Investments"].fillna(0),

        NCAV=lambda d: d["Total Current Assets"] - d["Total Liabilities"],
        NCAV_Inv=lambda d: (
            d["Total Current Assets"]
            - d["Total Liabilities"]
            + d["Total_Investments_filled"]
        ),
        Book_Value=lambda d: d["Total Assets"] - d["Total Liabilities"],
        Tangible_Book_Value=lambda d: (
            d["Total Assets"]
            - d["Total Liabilities"]
            - d["Total Goodwill and Intangibles"].fillna(0)  # 0 if no goodwill reported
        ),

        # Ratios: NaN Market Cap or denominator => NaN => "N/A" after format_ratio
        Price_Book_Ratio=lambda d: d["Market Cap"] / d["Book_Value"],
        Price_Tangible_Book_Ratio=lambda d: d["Market Cap"] / d["Tangible_Book_Value"],
        Price_NCAV_Ratio=lambda d: d["Market Cap"] / d["NCAV"],
        Price_NCAV_Inv_Ratio=lambda d: d["Market Cap"] / d["NCAV_Inv"],
    )
    .drop(columns=["Total_Investments_filled"])  # cleanup helper column
)

# --- Apply formatting (on a copy so raw numeric df is preserved) ---
display_df = formatted_df.copy()

currency_cols = [
    "Market Cap", "Total Assets", "Total Current Assets", "Total Liabilities",
    "Total Investments", "Total Goodwill and Intangibles",
    "NCAV", "NCAV_Inv", "Book_Value", "Tangible_Book_Value",
    "Latest FY Net Income", "TTM Net Income",
]
ratio_cols = [
    "Price_Book_Ratio", "Price_Tangible_Book_Ratio",
    "Price_NCAV_Ratio", "Price_NCAV_Inv_Ratio",
]

for col in currency_cols:
    if col in display_df.columns:
        display_df[col] = display_df[col].map(format_currency)

for col in ratio_cols:
    if col in display_df.columns:
        display_df[col] = display_df[col].map(format_ratio)

display_df

Fetching data for: 2267.T...
Fetching data for: 3435.T...
Fetching data for: 1846.HK...
Fetching data for: 6889.HK...
   ⚠️  Currency mismatch (HKD vs JPY). Fetching FX rate HKDJPY=X...


c:\Users\alexa\AppData\Local\Programs\Python\Python310\lib\site-packages\yfinance\scrapers\history.py:389: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  self._capital_gains = pd.Series()


   Converted Market Cap at rate: 20.2819
Fetching data for: 7399.T...
Fetching data for: 1913.HK...
   ⚠️  Currency mismatch (HKD vs EUR). Fetching FX rate HKDEUR=X...


c:\Users\alexa\AppData\Local\Programs\Python\Python310\lib\site-packages\yfinance\scrapers\history.py:389: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  self._capital_gains = pd.Series()


   Converted Market Cap at rate: 0.1100
Fetching data for: VTU.L...


,Ticker,Trading Currency,Financial Currency,Market Cap,Total Assets,Total Current Assets,Total Goodwill and Intangibles,Total Liabilities,Total Investments,Latest FY Net Income,TTM Net Income,NCAV,NCAV_Inv,Book_Value,Tangible_Book_Value,Price_Book_Ratio,Price_Tangible_Book_Ratio,Price_NCAV_Ratio,Price_NCAV_Inv_Ratio
0,2267.T,JPY,JPY,814.6b,817.1b,315.8b,10.2b,210.4b,N/A,45.5b,N/A,105.4b,105.4b,606.7b,596.6b,1.34,1.37,7.72,7.72
1,3435.T,JPY,JPY,10.8b,26.2b,15.6b,100.8m,7.3b,N/A,1.1b,N/A,8.3b,8.3b,18.9b,18.8b,0.57,0.57,1.30,1.30
2,1846.HK,HKD,HKD,869.0m,2.0b,776.7m,400.0m,667.8m,N/A,54.5m,N/A,108.9m,108.9m,1.3b,884.5m,0.68,0.98,7.98,7.98
3,6889.HK,HKD,JPY,44.6b,359.3b,53.4b,6.8b,227.9b,6.7b,4.0b,N/A,-174.5b,-167.8b,131.4b,124.5b,0.34,0.36,N/A,N/A
4,7399.T,JPY,JPY,3.7b,14.3b,9.6b,131.9m,2.6b,N/A,212.7m,N/A,7.0b,7.0b,11.7b,11.6b,0.32,0.32,0.53,0.53
5,1913.HK,HKD,EUR,9.9b,11.0b,3.0b,1.9b,6.3b,N/A,851.9m,N/A,-3.3b,-3.3b,4.7b,2.8b,2.13,3.59,N/A,N/A
6,VTU.L,GBP,GBP,195.5m,1.6b,976.2m,136.6m,1.2b,N/A,14.6m,N/A,-225.4m,-225.4m,357.5m,221.0m,0.55,0.88,N/A,N/A
